In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from scipy.stats import t
import os

In [2]:
# Define the model functions
def mole_fraction_co2_model(res_time, mole_fraction_co2_eq, k):
    return mole_fraction_co2_eq - (mole_fraction_co2_eq - 1) * np.exp(-k * res_time)

In [3]:
def conv_model(res_time, conv_eq, k):
    return conv_eq - conv_eq * np.exp(-k * res_time)

In [4]:
# Constants and helper functions for energy efficiency calculation
def gibbs_free_enthalpy_CO(temp):
    return -1.48747E-20 * temp**6 + 2.90543E-16 * temp**5 - 2.18412E-12 * temp**4 + \
           7.80039E-09 * temp**3 - 1.14364E-05 * temp**2 - 8.19286E-02 * temp - 1.12455E+02

In [5]:
def gibbs_free_enthalpy_CO2(temp):
    return 1.21850E-21 * temp**6 - 2.63973E-17 * temp**5 + 2.25893E-13 * temp**4 - \
           9.56091E-10 * temp**3 + 2.75906E-06 * temp**2 - 4.68102E-03 * temp - 3.93205E+02

In [6]:
def fit_models(group):
    params_mole_fraction_co2, cov_mole_fraction_co2 = curve_fit(
        mole_fraction_co2_model,
        group['res_time_sec'],
        group['mole_fraction_co2'],
        p0=[0.5, 0.05]
    )
    params_conv, cov_conv = curve_fit(
        conv_model,
        group['res_time_sec'],
        group['conv'],
        p0=[0.5, 0.05]
    )
    return params_mole_fraction_co2, cov_mole_fraction_co2, params_conv, cov_conv

In [7]:
def calculate_confidence_intervals(
        res_time_range,
        params_mole_fraction_co2,
        cov_mole_fraction_co2,
        params_conv,
        n
):
    alpha = 0.05
    dof = max(0, n - len(params_mole_fraction_co2))
    tval = t.ppf(1.0 - alpha / 2., dof)
    
    mole_fraction_co2_eq, k = params_mole_fraction_co2
    mole_fraction_co2_eq_sd, k_sd = np.sqrt(np.diag(cov_mole_fraction_co2))
    
    conf_values = pd.DataFrame({'res_time_sec': res_time_range})
    conf_values['mole_fraction_co2_fit'] = mole_fraction_co2_model(res_time_range, mole_fraction_co2_eq, k)
    conf_values['conv_fit'] = conv_model(res_time_range, params_conv[0], params_conv[1])
    
    conf_values['mole_fraction_co2_fit_sd'] = np.sqrt(
        (mole_fraction_co2_eq_sd**2 + (1 - np.exp(-k * res_time_range))**2 * k_sd**2)
    )
    conf_values['mole_fraction_co2_fit_lcl'] = conf_values['mole_fraction_co2_fit'] - tval * conf_values['mole_fraction_co2_fit_sd']
    conf_values['mole_fraction_co2_fit_ucl'] = conf_values['mole_fraction_co2_fit'] + tval * conf_values['mole_fraction_co2_fit_sd']
    
    conv_eq = (1 - mole_fraction_co2_eq) / (1 + 1 / 2 * mole_fraction_co2_eq)
    conv_eq_sd = conv_eq * np.sqrt(
        (mole_fraction_co2_eq_sd / (1 - mole_fraction_co2_eq))**2 +
        ((0.5 * mole_fraction_co2_eq_sd) / (1 + 0.5 * mole_fraction_co2_eq))**2
    )
    conf_values['conv_fit_sd'] = np.sqrt(
        (conv_eq_sd / conv_eq)**2 + ((conv_eq * (1 - np.exp(-k * res_time_range)))**2 * k_sd**2)
    )
    conf_values['conv_fit_lcl'] = conf_values['conv_fit'] - tval * conf_values['conv_fit_sd']
    conf_values['conv_fit_ucl'] = conf_values['conv_fit'] + tval * conf_values['conv_fit_sd']
    
    return conf_values, mole_fraction_co2_eq, mole_fraction_co2_eq_sd, conv_eq, conv_eq_sd, k, k_sd

In [8]:
def energy_efficiency_calculations(conf_values, group):
    power_avg = group['plasma_power_watt_avg'].mean()
    reactor_vol_ml = 17.31
    packing_factor = 0.4774
    conc_co2_blank = 1
    gas_temp_avg = group['gas_temp_avg_dgc'].mean()
    gas_temp = 273.15 + gas_temp_avg
    
    gf_co = gibbs_free_enthalpy_CO(gas_temp)
    gf_co2 = gibbs_free_enthalpy_CO2(gas_temp)
    
    conf_values['co2_flux_fit'] = (reactor_vol_ml * (1 - packing_factor)) / (conf_values['res_time_sec'] / 60)
    conf_values['alpha_fit'] = 1 + (1 / 2) * conf_values['conv_fit']
    conf_values['alpha_fit_sd'] = (1 / 2) * conf_values['conv_fit'] * conf_values['conv_fit_sd'] / conf_values['conv_fit']
    
    conf_values['conc_co_fit'] = conf_values['conv_fit'] / (1 + (1 / 2) * conf_values['conv_fit'])
    conf_values['conc_co_fit_sd'] = conf_values['conc_co_fit'] * np.sqrt(
        conf_values['conv_fit_sd']**2 / conf_values['conv_fit']**2 +
        (0.5 * conf_values['conv_fit_sd'] / (1 + 0.5 * conf_values['conv_fit']))**2
    )
    
    conf_values['conc_o2_fit'] = conf_values['conv_fit'] / (2 + conf_values['conv_fit'])
    conf_values['conc_o2_fit_sd'] = conf_values['conc_o2_fit'] * np.sqrt(
        conf_values['conv_fit_sd']**2 / conf_values['conv_fit']**2 +
        (conf_values['conv_fit_sd'] / (2 + conf_values['conv_fit']))**2
    )
    
    conf_values['sei_fit'] = (power_avg / conf_values['co2_flux_fit']) * 60 * 24.055
    
    conf_values['ee_gf_fit'] = (conf_values['alpha_fit'] * conf_values['conc_co_fit'] * gf_co -
                                conf_values['conv_fit'] * conc_co2_blank * gf_co2) / conf_values['sei_fit'] * 100
    conf_values['ee_gf_fit_sd'] = 100 * np.sqrt(
        (conf_values['alpha_fit_sd'] / conf_values['alpha_fit'] * conf_values['alpha_fit'] * conf_values['conc_co_fit'] * gf_co)**2 +
        (conf_values['conv_fit_sd'] * conc_co2_blank * gf_co2)**2
    ) / conf_values['sei_fit']
    
    conf_values['ee_gf_fit_lcl'] = conf_values['ee_gf_fit'] - t.ppf(1.0 - 0.05 / 2., len(group) - 1) * conf_values['ee_gf_fit_sd']
    conf_values['ee_gf_fit_ucl'] = conf_values['ee_gf_fit'] + t.ppf(1.0 - 0.05 / 2., len(group) - 1) * conf_values['ee_gf_fit_sd']
    
    return conf_values

In [9]:
# Load the data
file_path = r'N:\FWET\FDCH\AdsCatal\General\personal_work_folders\plasmacatdesign\co2-splitting\uhasselt\uhasselt_co2_spl_reaction_data_combined.csv'
data = pd.read_csv(file_path)

# Filter data for 'CO2' compound and calculate mole_fraction_co2
data = data[data['compound'] == 'CO2']
data['mole_fraction_co2'] = (1 - data['conv']) / (1 + 1 / 2 * data['conv'])

# Group by 'packing' and perform the fitting for each group
all_results = []
res_time_range = np.linspace(1, 100, 199)

for packing, group in data.groupby('packing'):
    params_mole_fraction_co2, cov_mole_fraction_co2, params_conv, cov_conv = fit_models(group)
    conf_values, mole_fraction_co2_eq, mole_fraction_co2_eq_sd, conv_eq, conv_eq_sd, k, k_sd = calculate_confidence_intervals(
        res_time_range, params_mole_fraction_co2, cov_mole_fraction_co2, params_conv, len(group))
    conf_values = energy_efficiency_calculations(conf_values, group)
    
    conf_values['packing'] = packing
    conf_values['mole_fraction_co2_eq'] = mole_fraction_co2_eq
    conf_values['mole_fraction_co2_eq_sd'] = mole_fraction_co2_eq_sd
    conf_values['conv_eq'] = conv_eq
    conf_values['conv_eq_sd'] = conv_eq_sd
    conf_values['k'] = k
    conf_values['k_sd'] = k_sd
    
	# Merge the original 'conv' values with the fitted results
    conf_values = pd.merge(
        conf_values,
        group[['packing', 'res_time_sec', 'conv']],
        on='res_time_sec',
        how='left'
    )

    all_results.append(conf_values)

# Combine all results into a single DataFrame
final_results = pd.concat(all_results, ignore_index=True)

# Save the results to CSV
output_path = os.path.join(
    r'N:\FWET\FDCH\AdsCatal\General\personal_work_folders\plasmacatdesign\co2-splitting\uhasselt',
    'uhasselt_co2_spl_reaction_fitting_results.csv'
)
final_results.to_csv(output_path, index=False)

print("Fitting results saved to:", output_path)

Fitting results saved to: N:\FWET\FDCH\AdsCatal\General\personal_work_folders\plasmacatdesign\co2-splitting\uhasselt\uhasselt_co2_spl_reaction_fitting_results.csv
